# Main for Energy Simulation 

Simulates the energy consumption for each track, calculates the SoC each truck would have and calculates the resulting load profiles for the freight forwarding locations 

This notebook is to be executed after data/data_handling/data-aquisition.ipynb and main_data_analysis.ipynb

### Imports 

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import seaborn as sns
import sys
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches 
import matplotlib.lines as mlines
import matplotlib.colors
import matplotlib.ticker as mticker

from utils.style_config import *

sys.path.append('energy_sim')
import data_handling.data_processing as dp
import energy_sim.sequential_analysis as sq
import energy_sim.load_profile as lp

### Style parameters

In [ ]:
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'sans-serif',
    'font.sans-serif': 'Arial',
    'font.size': 12,
    'text.latex.preamble': r'\usepackage{siunitx}'
})

### Scenario Parameters

In [ ]:
# Constant charging powers 
params_set = pd.DataFrame(
   {'default':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 150, 'industrial area': 0},
         },
          'big_batt':{'batt_cap': 798,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 150, 'industrial area': 0},
         },
          'destination':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 150, 'industrial area': 350},
         },
          'low_home':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 50, 'industrial area': 0},
         },
          'high_home':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 350, 'industrial area': 0},
         },
          'ultra_home':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 10000, 'industrial area': 0},
         },
}
)
display(params_set)

# set one default scenario
DEFAULT_SCENARIO = 'default'

## Perform Energy Simulation based on BET.OS script 

The energy simulation takes very long to run (in total approx. 24 h) and should just only be executed once 

First run the faulty_tracks() function to identify tracks with missing data (results will be saved in mismatched_tracks.csv)
Then run run_energy_sim() to calculate energy consumption for each track (results will be saved in tracks_filtered_with_energy.csv)
Run run_energy_sim() will additionally create zero_speed_tracks.csv which lists tracks that have no speed data

The outputs are saved as csvs

### Run once:

In [ ]:
#faulty_tracks()

# trips_energy = run_energy_sim(override_mismatched_tracks=False)

# "Can either pass on trips_energy directly from the simulation or let the function read in tracks_with_energy_raw.csv"
# df_trip_energy = clean_energy_sim_data(trips=None)

### After running the energy simulation once, read the csv instead of excecuting the simulation

In [ ]:
df_trips = pd.read_csv('input/stations/tracks_filtered.csv', index_col='track_id')
df_trip_energy= pd.read_csv('output/csvs/tracks_with_energy.csv', index_col='track_id')
if False: # debugging
    df_trip_energy = df_trip_energy.head(10000)
    df_trips = df_trips.loc[df_trips.index.to_series().isin(df_trip_energy.index)]

df_trips['stop_time'] = pd.to_datetime(df_trips['stop_time'], format='ISO8601', utc=True)
df_trips['start_time'] = pd.to_datetime(df_trips['start_time'], format='ISO8601', utc=True)
df_stops = dp.process_stops_data(df_trips.drop(columns=['duration', 'max_speed_kmh', 'avg_speed_kmh', 'gross_vehicle_weight', 'total_mass_with_trailer', 'axle_class',
                       "avg_speed", "max_speed", "n_signal_loss", "d_signal_loss", "r_signal_loss", "avg_hdop"]))
df_occ_energy = sq.combine_tracks_and_stops(df_stops = df_stops, df_tracks_with_energy = df_trip_energy)


In [ ]:
soc_results = {}
tours_energy_results = {}
results = {}
for scenario in params_set.columns:
    params = params_set[scenario].to_dict()
    print("")
    print(f"******************************")
    print(f"Running scenario: {scenario}")
    df_soc = sq.truck_soc(df_activities = df_occ_energy, **params)
    results_dict = sq.evaluate_charging_distribution(df_soc, evaluate_per_fleet=False, **params)
    results.update({scenario: results_dict})
    
    # Neu: Berechne und speichere df_tours_energy pro Szenario
    df_tours_energy_scenario = dp.aggregate_tours(df_soc, energy=True)
    tours_energy_results[scenario] = df_tours_energy_scenario
    # Speichere df_soc pro Szenario
    soc_results[scenario] = df_soc
    
df_results = pd.DataFrame(results).T

In [ ]:
df_charging_per_fleet = soc_results['destination'].groupby(['freight_forwarder', 'occupation']).energy_recharged_kwh.sum().unstack().drop(columns=['other area', 'rest area', 'service area fuel'])

df_charging_per_fleet = (df_charging_per_fleet.T / df_charging_per_fleet.T.sum()).T

df_charging_per_fleet.plot.bar(stacked=True, figsize=(10, 6))

display(df_charging_per_fleet)
print((100*df_charging_per_fleet).to_latex(float_format="\SI{%.2f}{\percent}", label="df_charging_per_fleet", caption="Distribution of charging energy per freight forwarder and occupation. The values are normalized to the total energy recharged per freight forwarder.", index=True, column_format='l' + 'c' * (len(df_charging_per_fleet.columns) - 1)))

In [ ]:
df_results.plot(
    kind='bar',
    stacked=True,
    color=[line_colors['TUMBlue1'], line_colors['Purple'], line_colors['TUMOrange']],
    figsize=(8, 4)
)
plt.ylabel('Proportion')
plt.xlabel('Scenario')
plt.title('Stacked Barplot of Charging Location Proportions per Scenario')
plt.legend(title='Location')
plt.tight_layout()
plt.show()
df_results.to_csv('output/csvs/charging_distribution.csv')

In [ ]:
params = params_set[DEFAULT_SCENARIO].to_dict()

threshold = params['batt_cap'] * ( params['soc_max'] - params['soc_min'] )

### Tour statistics

In [ ]:
"""
In contrast to df_tours, df_tours_energy also includes non-driving activities between tracks. 
Thus, some additional parameters are included

    Stop time is the stop time of the last track equivalently to df_tours, 
    End time additionally includes the time spent at the home base after the tour's last track (i.e. = start time of the next tour)
   
    driving_duration is the sum of the duration of all driving activities in the tour, equivalently to duration in df_tours
    duration is the time between the start of the tour's first track and the end of its last activity 
     (i.e the duration between the start of the tour and the start of the next tour)
"""

df_tours_energy = dp.aggregate_tours(soc_results[DEFAULT_SCENARIO], energy=True)
df_tours_energy

In [ ]:
# -------------------------------------------------------------------------------------------
#                           SOC BY FREIGHT FORWARDER VIOLIN PLOTS (ENERGY)
# -------------------------------------------------------------------------------------------

def plot_soc_min_violin(tours_energy_results, cut=-1):
    """
    Create a single violin plot showing the distribution of soc_no_public_charging_min for each freight forwarder.
    Overlays three scenarios per forwarder: 50kW (light transparent), 150kW (solid), 350kW (medium transparent).
    """
    # Extract and prepare data for the three scenarios
    scenarios = ['low_home', 'default', 'high_home']  # Adjusted order to plot high_home last for visibility of its lines
    dfs = []
    for sc in scenarios:
        if sc in tours_energy_results:
            df_sc = tours_energy_results[sc].copy()
            df_sc['scenario'] = sc  # Add new column for scenario
            dfs.append(df_sc)
    
    if not dfs:
        raise ValueError("No matching scenarios found in tours_energy_results.")
    
    plot_df = pd.concat(dfs, ignore_index=True)
    
    # Adjust colors: Based on palette_ff, but with transparency
    # Create a custom palette with alpha values
    alpha_map = {'low_home': 0.3, 'default': 1.0, 'high_home': 0.6}
    unique_ff = sorted(plot_df['freight_forwarder'].unique())
    palette_adjusted = {}
    for ff in unique_ff:
        base_color = palette_ff[ff]  # Assume palette_ff is a dict with colors per FF
        palette_adjusted[ff] = {}  # Nested for scenarios
        for sc in scenarios:
            rgba = list(matplotlib.colors.to_rgba(base_color))
            rgba[3] = alpha_map.get(sc, 1.0) 
            palette_adjusted[ff][sc] = tuple(rgba)
    
    # Since sns.violinplot does not directly support nested palettes, plot per scenario separately and overlay
    plt.figure(figsize=(textwidth, h_169))
    for sc in scenarios:
        df_sc = plot_df[plot_df['scenario'] == sc]
        num_col_before = len(plt.gca().collections)
        sns.violinplot(
            data=df_sc,
            x='freight_forwarder',
            y='soc_no_public_charging_min',
            hue='freight_forwarder',  # Hue on FF to maintain colors per FF
            palette={ff: palette_adjusted[ff][sc] for ff in unique_ff},
            inner='quartile' if sc == 'default' else None,  # Show horizontal lines only for default (150kW)
            cut=cut,
            linewidth=1.5 if sc == 'default' else 1.0,  # Thicker line for default
            bw_adjust=0.5,
            dodge=False,  # Overlay instead of side-by-side
            legend=False,
            alpha=alpha_map.get(sc, 1.0)  # Alpha for the fill (combined with color)
        )
        if sc == 'low_home':
            for i in range(num_col_before, len(plt.gca().collections)):
                plt.gca().collections[i].set_hatch('//')
                plt.gca().collections[i].set_edgecolor('black')
                plt.gca().collections[i].set_linewidth(0.5)
        if sc == 'high_home':
            for i in range(num_col_before, len(plt.gca().collections)):
                plt.gca().collections[i].set_linestyle('--')  # Make outer lines dashed for 350kW
                plt.gca().collections[i].set_linewidth(1.0)
    
    plt.grid(True, axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
    plt.xlabel('Freight Forwarder')
    plt.ylabel('Minimum State of Charge (SoC) / \%')
    plt.ylim(cut, 1)
    plt.gca().yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    # plt.title('Distribution of Minimum SoC per Tour by Charging Power')
    plt.xticks(
        ticks=range(len(unique_ff)),
        labels=unique_ff
    )
    
    # Add dashed line at the threshold (assuming y=0 as the physical threshold for SoC, to indicate clipped area and outliers)
    plt.axhline(y=0, linestyle='--', color='black', alpha=0.7, linewidth=1.2, label='0% SoC Threshold (clipped below)')
    
    # Add legend at the bottom, horizontal and flat
    # Use a neutral color for legend patches to represent structure for all freight forwarders
    legend_colors = {
        'low_home': 'gray',  # Darker gray for better visibility
        'default': 'darkgray',  # Strong, solid color
        'high_home': 'lightgray'  # Distinct and visible
    }
    power_map = {'low_home': '50', 'default': '150', 'high_home': '350'}
    handles = [
        mpatches.Patch(color=legend_colors[sc], alpha=min(1.0, alpha_map[sc] + 0.2), hatch='//' if sc == 'low_home' else None, label=f'{power_map[sc]} kW') 
        if sc != 'high_home' else
        mlines.Line2D([], [], color=legend_colors[sc], alpha=min(1.0, alpha_map[sc] + 0.2), linestyle='--', linewidth=2, label=f'{power_map[sc]} kW')
        for sc in scenarios
    ]
    plt.legend(handles=handles, title='Home Charging Power', loc='lower center', bbox_to_anchor=(0.5, -0.02),
               ncol=3, borderaxespad=0., frameon=False)
    
    plt.tight_layout()
    plt.savefig('output/figures/soc_min_violin.svg')

plot_soc_min_violin(tours_energy_results, cut=-1)

## Charging Energy Demand per Home Base

In [ ]:
daily_demands = dp.daily_energy_demands(df_tours_energy, threshold, params['charging_powers'])

print(daily_demands.head())

# Most critical for ff 4. Where we have a representative sample of 10% of all of the ff's trucks --> daily load profiles would be 10x of these results
# 90/135 or 66 % of all recorded days could not be served with one traffo even if we assume an even load profile throughout the day 
# (max recharge in a day with 1 traffo = 630 kW * 24h = 15120 kWh)

In [ ]:

# ------------------------------------------------------------------------------
#                             WEEKLY ENERGY DEMAND BOXPLOT
# ------------------------------------------------------------------------------


def plot_weekly_energy_demand_boxplot(df_loads):
    """
    Creates a figure with a central boxplot showing all home bases' energy demand together
    at the top, and individual plots for each home base (CID) below.
    
    Parameters:plot_weekly_energy_demand_boxplot(daily_demands)
    -----------
    df_loads : DataFrame
        DataFrame containing daily energy demand data with day, energy_demand_kwh, cid, and freight_forwarder columns
    """
    
    # Convert day column to datetime and extract weekday
    df_loads['day'] = pd.to_datetime(df_loads['day'])
    df_loads['weekday'] = df_loads['day'].dt.weekday  # 0 = Monday, 6 = Sunday

    # Create subplots with a 3 column x 5 row grid (15 subplots total)
    fig = plt.figure(figsize=(textwidth, h_169))
    
    # Add a GridSpec with 5 rows and 3 columns
    gs = fig.add_gridspec(5, 3, height_ratios=[2, 1, 1, 1, 1])
    
    # Add the central plot at the top, spanning all columns
    central_ax = fig.add_subplot(gs[0, :])
    
    # Create the individual CID axes
    axes = []
    for i in range(1, 5):  # Rows 1-4
        for j in range(3):  # 3 columns
            ax = fig.add_subplot(gs[i, j])
            axes.append(ax)
    
    # Get unique CIDs
    cids = sorted(df_loads['cid'].unique())
    
    # Prepare data for the central plot
    all_boxplot_data = []
    all_weekday_labels = []
    all_weekday_numbers = []  # Store the actual weekday numbers
    
    for weekday in range(7):
        weekday_data = df_loads[df_loads['weekday'] == weekday]['energy_demand_kwh'].tolist()
        if len(weekday_data) > 0:  # Only add if there's data for this weekday
            all_boxplot_data.append(weekday_data)
            all_weekday_labels.append(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][weekday])
            all_weekday_numbers.append(weekday)
    
    # Add the average data for the central plot (all weekdays combined)
    all_avg_data = df_loads['energy_demand_kwh'].tolist()
    all_boxplot_data.append(all_avg_data)
    all_weekday_labels.append('Combined')
    all_weekday_numbers.append('Combined')
    
    # Create central boxplot with all data
    scaling_factor = textwidth / 16
    bp_central = central_ax.boxplot(
        all_boxplot_data,
        patch_artist=True,
        showmeans=True,
        meanline=True,
        meanprops=dict(color=colors['TUMGreen3'], linestyle='--', linewidth=1.5 * scaling_factor),
        medianprops=dict(color=colors['TUMOrange'], linewidth=1.5 * scaling_factor),
        showfliers=True,
        flierprops=dict(marker='o', color='red', alpha=0.5, markersize=3),
        whiskerprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
        capprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
        widths=0.55 * scaling_factor,
        positions=range(len(all_boxplot_data)),
        labels=all_weekday_labels
    )
    
    # Style central boxplot
    central_ax.set_title('All Home Bases Combined')
    central_ax.tick_params(axis='x', )
    central_ax.tick_params(axis='y', )
    central_ax.grid(True, which='both', axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
    central_ax.set_ylim(bottom=0)
    
    # Define legend handles directly in the central plot
    mean_line = mlines.Line2D([], [], color=colors['TUMGreen3'], linestyle='--', label='Mean', linewidth=3 * scaling_factor)
    median_line = mlines.Line2D([], [], color=colors['TUMOrange'], label='Median', linewidth=3 * scaling_factor)
    # Add the legend to the top left corner of the central plot
    central_ax.legend(handles=[median_line, mean_line], loc='upper left',  frameon=True, ncol=2)
    
    # Assign colors to the central boxplots based on weekday or 'Combined'
    for i, (box, weekday_or_combined) in enumerate(zip(bp_central['boxes'], all_weekday_numbers)):
        box.set_facecolor(palette_weekdays[weekday_or_combined])
        # Add a slightly thicker border to the average box to make it stand out
        if weekday_or_combined == 'Combined':  # Average box
            box.set(linewidth=2.0 * scaling_factor)
    
    # Annotate central plot median values
    for n, (median_feature, demands) in enumerate(zip(bp_central['medians'], all_boxplot_data)):
        demands_series = pd.Series(demands)
        median_value = demands_series.median()
        x_median, y_median = median_feature.get_xydata()[1]
        central_ax.text(x_median, y_median, f'{median_value:.2f}', 
                horizontalalignment='center', color=colors['TUMBlack'],
                bbox=dict(boxstyle='round', pad=0.3 * scaling_factor, facecolor=colors['TUMWhite'], 
                        edgecolor=colors['TUMOrange'], alpha=0.9))
    
    # Process each CID for individual plots
    for i, cid in enumerate(cids):
        if i >= len(axes):
            break
            
        # Filter data for current CID
        cid_data = df_loads[df_loads['cid'] == cid]
        
        # Get the freight forwarder for this CID
        ff = cid_data['freight_forwarder'].iloc[0] if not cid_data.empty else "Unknown"
        
        # Calculate and print daily energy demand per home base
        print(f"\nHome Base CID {cid} - Daily Energy Demand:")
        weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        daily_stats = cid_data.groupby('weekday')['energy_demand_kwh'].agg(['mean', 'std', 'median', 'count']).reset_index()
        daily_stats['weekday_name'] = daily_stats['weekday'].apply(lambda x: weekday_names[x])
        print(daily_stats[['weekday_name', 'mean', 'std', 'median', 'count']])
        print(f"Overall mean: {cid_data['energy_demand_kwh'].mean():.2f} kWh")
        
        # Prepare boxplot data with only available weekdays
        boxplot_data = []
        weekday_labels = []
        weekday_numbers = []  # Store the actual weekday numbers
        
        for weekday in range(7):
            weekday_data = cid_data[cid_data['weekday'] == weekday]['energy_demand_kwh'].tolist()
            if len(weekday_data) > 0:  # Only add if there's data for this weekday
                boxplot_data.append(weekday_data)
                weekday_labels.append(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][weekday])
                weekday_numbers.append(weekday)
        
        # Add the average data for this CID (all weekdays combined)
        avg_data = cid_data['energy_demand_kwh'].tolist()
        boxplot_data.append(avg_data)
        weekday_labels.append('Combined')
        weekday_numbers.append('Combined')
        
        # Create boxplot
        bp = axes[i].boxplot(
            boxplot_data,
            patch_artist=True,
            showmeans=True,
            meanline=True,
            meanprops=dict(color=colors['TUMGreen3'], linestyle='--', linewidth=1.5 * scaling_factor),
            medianprops=dict(color=colors['TUMOrange'], linewidth=1.5 * scaling_factor),
            showfliers=True,
            flierprops=dict(marker='o', color='red', alpha=0.5, markersize=3),
            whiskerprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
            capprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
            widths=0.55 * scaling_factor,
            positions=range(len(boxplot_data)),
            labels=weekday_labels
        )
        
        # Set x-axis and y-axis tick labels font size
        axes[i].tick_params(axis='x', labelsize=10 * scaling_factor)
        axes[i].tick_params(axis='y', labelsize=10 * scaling_factor)
        
        # Annotate median values
        for n, (median_feature, demands) in enumerate(zip(bp['medians'], boxplot_data)):
            demands_series = pd.Series(demands)
            median_value = demands_series.median()
            x_median, y_median = median_feature.get_xydata()[1]
            axes[i].text(x_median, y_median, f'{median_value:.2f}', 
                    horizontalalignment='center', color=colors['TUMBlack'],
                    bbox=dict(boxstyle='round', pad=0.3 * scaling_factor, facecolor=colors['TUMWhite'], 
                            edgecolor=colors['TUMOrange'], alpha=0.9))
        
        # Assign colors to the boxplots based on weekday or 'Combined'
        for j, (box, weekday_or_combined) in enumerate(zip(bp['boxes'], weekday_numbers)):
            box.set_facecolor(palette_weekdays[weekday_or_combined])
            # Add a slightly thicker border to the average box to make it stand out
            if weekday_or_combined == 'Combined':  # Average box
                box.set(linewidth=2.0 * scaling_factor)
        
        # Set title for each subplot with both freight forwarder and CID
        axes[i].set_title(f'Freight Forwarder {ff} - Home Base CID {cid}')
        
        # Add grid
        axes[i].grid(True, which='both', axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
        
        # Set y-axis limit
        axes[i].set_ylim(bottom=0)
        
        # Customize spines
        for spine in axes[i].spines.values():
            spine.set_edgecolor(colors['TUMBlack'])
            spine.set_linewidth(0.8 * scaling_factor)
    
    # Remove any empty subplots if there are fewer than 14 CIDs
    for i in range(len(cids), len(axes)):
        fig.delaxes(axes[i])
    
    # Add common labels
    fig.text(0.01, 0.5, 'Energy Demand / kWh', ha='center', va='center', rotation='vertical')

    plt.tight_layout(rect=[0.02, 0.03, 1, 0.95])  # Adjust layout to make room for common labels
    # plt.savefig('output/figures/energy/home_base_weekday_energy_boxplots.pdf', bbox_inches='tight')
    # plt.show()
    
# plot_weekly_energy_demand_boxplot(daily_demands)

## Create Load Profiles

In [ ]:
# ------------------------------------------------------------------------------
#                             LOAD PROFILE CALCULATION
# ------------------------------------------------------------------------------
df_charging_stats = pd.DataFrame()
df_load_profiles = pd.DataFrame()
# Call the function with the load threshold parameter (default is 630 kW)
for scenario in params_set.columns:
    params = params_set[scenario].to_dict()
    print(f"Calculating load profiles for scenario: {scenario}")
    lp_, _, charging_stats = lp.calculate_charging_load_profiles(
        soc_results[scenario], charging_power=params['charging_powers']['home base'], load_threshold=630)
    
    lp_['scenario'] = scenario
    df_load_profiles = pd.concat([df_load_profiles, lp_.reset_index()], axis=0)

    # charging_stats is now a dict of dicts (cid -> stats), so transpose for DataFrame
    df_stats = pd.DataFrame.from_dict(charging_stats, orient='index')
    df_stats['scenario'] = scenario
    df_charging_stats = pd.concat([df_charging_stats, df_stats.reset_index(names='cid')], axis=0)

display(df_charging_stats)
display(df_load_profiles)

In [ ]:
# display(df_charging_stats[df_charging_stats['scenario'] == 'default'])

In [ ]:
# ------------------------------------------------------------------------------
#                             LOAD PROFILE PLOT - MINUTE RESOLUTION (OPTIMIZED)
# ------------------------------------------------------------------------------
# Selected CIDs
selected_cids = [412, 842, 809]
# selected_cids = df_charging_stats.cid.unique()

palette_grid_load = {
    'mean': colors['TUMBlue1'],
    'max': colors['TUMOrange'],
    'threshold': colors['TUMGreen3'],
    'threshold2': colors['TUMGreen2'],
}

linestyles_scenarios = {'default': '-', 
    'low_home': '--', 
    'high_home': ':', 
    'destination': '-.', 
    'big_batt': (0, (3, 1, 1, 1)),
    'ultra_home': (0, (5, 2)),
    'backup1': (0, (1, 1)),
    'backup2': (0, (4, 1, 2, 1))
    }

def plot_load_profiles_6plots(load_profiles, selected_cids):
    """
    Create a 6-plot figure: Left = max load profiles per location (CID),
    Right = 15min average load per location (CID).
    Uses freight forwarder color palette, scientific axis labels, and consistent styling.
    All font sizes set to 15 for professional presentation.
    """
    plt.rcParams['text.usetex'] = True
    plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
    plt.rcParams.update({'font.size': 15})  # Global font size 15

    # Use freight forwarder color palette from style_config
    colors_plot = [palette_ff[1], palette_ff[2], palette_ff[3], palette_ff[4], palette_ff[5], palette_ff[6]]
    
    # Define scenario colors with freight forwarder palette
    scenario_colors = {
        'default': colors_plot[0],      # TUMBlue1
        'low_home': colors_plot[1],     # TUMOrange
        'high_home': colors_plot[3],    # TUMGreen3
        'destination': colors_plot[2],  # Purple
        'big_batt': colors_plot[5],     # TUMBlue2
        'ultra_home': colors_plot[4]    # TUMBlue4
    }
    
    # Define markers for max load profiles
    markers_scenarios = {
        'default': 'o',      # Circle
        'low_home': 's',     # Square
        'high_home': '^',    # Triangle up
        'destination': 'D',  # Diamond
        'big_batt': 'v',     # Triangle down
        'ultra_home': 'p'    # Pentagon
    }
    
    # Define marker fill patterns
    marker_fill = {
        'default': 'full',        # Filled
        'low_home': 'none',       # Empty
        'high_home': 'full',      # Filled
        'destination': 'none',    # Empty
        'big_batt': 'full',       # Filled
        'ultra_home': 'none'      # Empty
    }
    
    # TUM blue colors for thresholds
    threshold_colors = {
        'threshold': colors['TUMBlue2'],   # TUMBlue2 for 630 kW
        'threshold2': colors['TUMBlue3']   # TUMBlue3 for 1260 kW
    }
    
    # Linestyles for average plots
    linestyles_scenarios = {
        'default': '-', 
        'low_home': '--', 
        'high_home': ':', 
        'destination': '-.', 
        'big_batt': (0, (3, 1, 1, 1)),
        'ultra_home': (0, (5, 2))
    }
    
    # Create figure with optimized layout
    fig, axes = plt.subplots(3, 2, figsize=(textwidth * 4.0, 2 * h_169 * 2), sharex='col')
    scenarios = load_profiles['scenario'].unique()
    
    # Calculate global maximum for y-axis scaling
    global_max = 0
    for scenario in scenarios:
        if scenario == 'ultra_home':
            continue
        df_scenario = load_profiles[load_profiles['scenario'] == scenario].copy()
        if 'time_' in df_scenario.columns:
            df_scenario.set_index('time_', inplace=True)
        
        for cid in selected_cids:
            if cid in df_scenario.columns:
                df_cid = df_scenario[[cid]].copy()
                df_cid = df_cid[df_cid[cid] > 0]
                if not df_cid.empty:
                    df_cid['time_of_day'] = df_cid.index.time
                    minute_max = df_cid.groupby('time_of_day')[cid].max()
                    max_load = minute_max.max() if not minute_max.empty else 0
                    global_max = max(global_max, max_load)
    
    # Set scaling for max plots
    global_max = global_max * 1.05
    
    for i, cid in enumerate(selected_cids):
        # LEFT: Max load profiles
        ax_left = axes[i, 0]
        ax_left.axhline(630, color=threshold_colors['threshold'], lw=3, label='630 kW threshold')
        ax_left.axhline(1260, color=threshold_colors['threshold2'], ls='--', lw=3, label='1260 kW threshold')
        ax_left.set_title(f'Location CID-{cid}: Maximum Load Profile', fontsize=15, fontweight='bold')
        ax_left.set_xticks(range(0, 25, 6))
        ax_left.xaxis.set_minor_locator(mticker.FixedLocator(range(0, 25, 1)))
        ax_left.set_xticklabels(['00:00', '06:00', '12:00', '18:00', '24:00'], fontsize=15)
        ax_left.set_xlim(0, 24)
        ax_left.set_ylim(0, global_max)
        ax_left.set_ylabel('Load / kW', fontsize=15)
        ax_left.tick_params(axis='both', which='major', labelsize=15)
        ax_left.grid('on', alpha=0.3)
        
        for scenario in scenarios:
            if scenario == 'ultra_home':
                continue
            df_cid = load_profiles[load_profiles['scenario'] == scenario].copy()
            if cid not in df_cid.columns:
                continue
            if 'time_' in df_cid.columns:
                df_cid.set_index('time_', inplace=True)
            df_cid = df_cid[[cid]].copy()
            df_cid = df_cid[df_cid[cid] > 0]
            if not df_cid.empty:
                df_cid['time_of_day'] = df_cid.index.time
                minute_max = df_cid.groupby('time_of_day')[cid].max()
                minute_max_hours = minute_max.index.map(lambda t: t.hour + t.minute / 60 + t.second / 3600)
                
                # Styling with freight forwarder colors - even thinner lines for max profiles
                if scenario == 'high_home':
                    ax_left.plot(minute_max_hours, minute_max.values, 
                                 label=f'{scenario_names[scenario]}',
                                 lw=1.0,
                                 color=scenario_colors[scenario],
                                 linestyle='-',
                                 marker=markers_scenarios[scenario],
                                 markersize=3,
                                 markevery=6,
                                 alpha=0.9,
                                 markerfacecolor='white' if marker_fill[scenario] == 'none' else scenario_colors[scenario],
                                 markeredgecolor=scenario_colors[scenario],
                                 markeredgewidth=1.2)
                else:
                    ax_left.plot(minute_max_hours, minute_max.values, 
                                 label=f'{scenario_names[scenario]}',
                                 lw=1.3,
                                 color=scenario_colors[scenario],
                                 linestyle='-',
                                 marker=markers_scenarios[scenario],
                                 markersize=4,
                                 markevery=6,
                                 markerfacecolor='white' if marker_fill[scenario] == 'none' else scenario_colors[scenario],
                                 markeredgecolor=scenario_colors[scenario],
                                 markeredgewidth=1.5)
        
        # RIGHT: 15min average load profiles
        ax_right = axes[i, 1]
        ax_right.axhline(630, color=threshold_colors['threshold'], lw=3, label='630 kW threshold')
        ax_right.axhline(1260, color=threshold_colors['threshold2'], ls='--', lw=3, label='1260 kW threshold')
        ax_right.set_title(f'Location CID-{cid}: 15-Minute Average Load', fontsize=15, fontweight='bold')
        ax_right.set_xticks(range(0, 25, 6))
        ax_right.xaxis.set_minor_locator(mticker.FixedLocator(range(0, 25, 1)))
        ax_right.set_xticklabels(['00:00', '06:00', '12:00', '18:00', '24:00'], fontsize=15)
        ax_right.set_xlim(0, 24)
        ax_right.set_ylim(0, 1000)
        ax_right.set_ylabel('Load / kW', fontsize=15)
        ax_right.tick_params(axis='both', which='major', labelsize=15)
        ax_right.grid('on', alpha=0.3)
        
        for scenario in scenarios:
            if scenario == 'ultra_home':
                continue
            df = load_profiles.loc[load_profiles.scenario==scenario,[cid,]]
            df['date'] = df.index.date
            df['15min'] = df.index.floor('15min')
            date_nonzero_energy = df.groupby('date')[cid].sum().gt(0)
            df = df.loc[df['date'].isin(date_nonzero_energy[date_nonzero_energy].index)]
            df_15min_load = df.groupby('15min')[cid].mean()
            plot_data = df_15min_load.groupby(df_15min_load.index.time).agg(['min', 'mean', 'max', 'median'])
            plot_data['hour'] = plot_data.index.map(lambda t: t.hour + t.minute / 60 + t.second / 3600)
            ax_right.step(x=plot_data.hour, y=plot_data['mean'], 
                          label=f'{scenario_names[scenario]}',
                          lw=2.2,
                          color=scenario_colors[scenario],
                          linestyle=linestyles_scenarios[scenario])
    
    # Add x-axis labels for bottom plots
    axes[2, 0].set_xlabel('Time / h', fontsize=15, fontweight='bold')
    axes[2, 1].set_xlabel('Time / h', fontsize=15, fontweight='bold')
    
    # Optimized layout
    plt.subplots_adjust(hspace=0.15, wspace=0.1)
    
    # Professional legend
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', title='Charging Scenarios', 
            bbox_to_anchor=(0.5, 0.98), ncol=6, fontsize=15, title_fontsize=15, frameon=False)
    
    return axes

def add_manual_zoom_area(ax, x_range=(8.0, 10.0), y_range=(300, 500), color='red'):
    """
    Draw a manual zoom area rectangle with professional styling
    """
    rect = plt.Rectangle((x_range[0], y_range[0]), 
                        x_range[1] - x_range[0], 
                        y_range[1] - y_range[0],
                        fill=False, edgecolor=color, linewidth=4, linestyle='--', zorder=10)
    ax.add_patch(rect)
    # Removed the "Zoom Area" text
    return rect

def create_zoom_view(load_profiles, selected_cids, cid_index, plot_type, x_range, y_range, scenario_colors, 
                    markers_scenarios, marker_fill, linestyles_scenarios, threshold_colors, scenario_names):
    """
    Create a magnified view of a specific area with freight forwarder colors and scientific labels
    """
    cid = selected_cids[cid_index]
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    if plot_type == 'max':
        ax.set_title(f'Location CID-{cid}: Maximum Load Profile\nZoom: {x_range[0]:.1f}-{x_range[1]:.1f} h, {y_range[0]}-{y_range[1]} kW', 
                    fontsize=16, fontweight='bold')
        scenarios = load_profiles['scenario'].unique()
        for scenario in scenarios:
            if scenario == 'ultra_home':
                continue
            df_cid = load_profiles[load_profiles['scenario'] == scenario].copy()
            if cid not in df_cid.columns:
                continue
            if 'time_' in df_cid.columns:
                df_cid.set_index('time_', inplace=True)
            df_cid = df_cid[[cid]].copy()
            df_cid = df_cid[df_cid[cid] > 0]
            if not df_cid.empty:
                df_cid['time_of_day'] = df_cid.index.time
                minute_max = df_cid.groupby('time_of_day')[cid].max()
                minute_max_hours = minute_max.index.map(lambda t: t.hour + t.minute / 60 + t.second / 3600)
                if scenario == 'high_home':
                    ax.plot(minute_max_hours, minute_max.values, 
                           label=f'{scenario_names[scenario]}',
                           lw=2.5, color=scenario_colors[scenario], linestyle='-',
                           marker=markers_scenarios[scenario], markersize=7, markevery=2,
                           alpha=0.9, markerfacecolor='white' if marker_fill[scenario] == 'none' else scenario_colors[scenario],
                           markeredgecolor=scenario_colors[scenario], markeredgewidth=2.0)
                else:
                    ax.plot(minute_max_hours, minute_max.values, 
                           label=f'{scenario_names[scenario]}',
                           lw=3.0, color=scenario_colors[scenario], linestyle='-',
                           marker=markers_scenarios[scenario], markersize=9, markevery=2,
                           markerfacecolor='white' if marker_fill[scenario] == 'none' else scenario_colors[scenario],
                           markeredgecolor=scenario_colors[scenario], markeredgewidth=2.5)
    elif plot_type == 'average':
        ax.set_title(f'Location CID-{cid}: 15-Minute Average Load\nZoom: {x_range[0]:.1f}-{x_range[1]:.1f} h, {y_range[0]}-{y_range[1]} kW', 
                    fontsize=16, fontweight='bold')
        scenarios = load_profiles['scenario'].unique()
        for scenario in scenarios:
            if scenario == 'ultra_home':
                continue
            df = load_profiles.loc[load_profiles.scenario==scenario,[cid,]]
            df['date'] = df.index.date
            df['15min'] = df.index.floor('15min')
            date_nonzero_energy = df.groupby('date')[cid].sum().gt(0)
            df = df.loc[df['date'].isin(date_nonzero_energy[date_nonzero_energy].index)]
            df_15min_load = df.groupby('15min')[cid].mean()
            plot_data = df_15min_load.groupby(df_15min_load.index.time).agg(['min', 'mean', 'max', 'median'])
            plot_data['hour'] = plot_data.index.map(lambda t: t.hour + t.minute / 60 + t.second / 3600)
            ax.step(x=plot_data.hour, y=plot_data['mean'], 
                   label=f'{scenario_names[scenario]}',
                   lw=2.5, color=scenario_colors[scenario], linestyle=linestyles_scenarios[scenario])
    
    # TUM blue thresholds
    ax.axhline(630, color=threshold_colors['threshold'], lw=3, label='630 kW threshold')
    ax.axhline(1260, color=threshold_colors['threshold2'], ls='--', lw=3, label='1260 kW threshold')
    ax.set_xlim(x_range[0], x_range[1])
    ax.set_ylim(y_range[0], y_range[1])
    ax.set_xlabel('Time / h', fontsize=15)
    ax.set_ylabel('Load / kW', fontsize=15)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.grid('on', alpha=0.3)
    
    # No legend for zoom plots
    # ax.legend(fontsize=14)  # Commented out
    
    return fig, ax

# Create the improved 6-plot visualization with freight forwarder colors
axes_6plots = plot_load_profiles_6plots(df_load_profiles.set_index('time_'), selected_cids)

# Find CID 842 index
cid_842_index = None
for i, cid in enumerate(selected_cids):
    if cid == 842:
        cid_842_index = i
        break

if cid_842_index is not None:
    # Add manual zoom areas to CID 842 plots - blue zoom area made larger
    zoom_area_1 = add_manual_zoom_area(axes_6plots[cid_842_index, 0], x_range=(8.0, 10.0), y_range=(200, 700), color='blue')
    zoom_area_2 = add_manual_zoom_area(axes_6plots[cid_842_index, 1], x_range=(8.5, 9.5), y_range=(10, 100), color='red')
    
    # Create zoom views with freight forwarder colors
    colors_plot = [palette_ff[1], palette_ff[2], palette_ff[3], palette_ff[4], palette_ff[5], palette_ff[6]]
    scenario_colors = {
        'default': colors_plot[0], 'low_home': colors_plot[1], 'high_home': colors_plot[3],
        'destination': colors_plot[2], 'big_batt': colors_plot[5], 'ultra_home': colors_plot[4]
    }
    markers_scenarios = {'default': 'o', 'low_home': 's', 'high_home': '^', 'destination': 'D', 'big_batt': 'v', 'ultra_home': 'p'}
    marker_fill = {'default': 'full', 'low_home': 'none', 'high_home': 'full', 'destination': 'none', 'big_batt': 'full', 'ultra_home': 'none'}
    linestyles_scenarios = {'default': '-', 'low_home': '--', 'high_home': ':', 'destination': '-.', 'big_batt': (0, (3, 1, 1, 1)), 'ultra_home': (0, (5, 2))}
    threshold_colors = {'threshold': colors['TUMBlue2'], 'threshold2': colors['TUMBlue3']}
    
    # Create zoom views with matching ranges to the zoom areas
    fig_zoom_max, ax_zoom_max = create_zoom_view(df_load_profiles.set_index('time_'), selected_cids, 
                                                cid_842_index, 'max', (8.0, 10.0), (200, 700),
                                                scenario_colors, markers_scenarios, marker_fill, 
                                                linestyles_scenarios, threshold_colors, scenario_names)

    fig_zoom_avg, ax_zoom_avg = create_zoom_view(df_load_profiles.set_index('time_'), selected_cids, 
                                                cid_842_index, 'average', (8.5, 9.5), (10, 100),
                                                scenario_colors, markers_scenarios, marker_fill, 
                                                linestyles_scenarios, threshold_colors, scenario_names)

    # Save zoom plots
    fig_zoom_max.savefig('output/figures/load_profiles/zoom_cid842_max_09h.pdf', bbox_inches='tight')
    fig_zoom_avg.savefig('output/figures/load_profiles/zoom_cid842_average_09h.pdf', bbox_inches='tight')

    plt.show()  # Show main plot
    fig_zoom_max.show()  # Show max zoom
    fig_zoom_avg.show()  # Show average zoom

# Save and show main plot
plt.tight_layout()
plt.savefig('output/figures/load_profiles/load_profiles_6plots_max_vs_15minavg.pdf', bbox_inches='tight')
plt.show()